# Create Time Series of Multiple Variables for a Given NERC Region


In [ ]:
# Start by importing the packages we need:
import os
import datetime
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Suppress Future Warnings


In [ ]:
# Suppress future warnings:
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)


## Set the Directory Structure

In [ ]:
# Identify the top-level directory and the subdirectory where the data will be stored:
temp_data_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/temperature_data/'
load_data_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/load_data/'
gridview_data_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/gridview_data/'
data_output_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/integrated_time_series/'


## Write a Function to Process the Temperature Time Series Data


In [ ]:
# Define a function to process the time series of temperature for a given NERC region:
def process_temperature_time_series(temp_data_dir: str, temp_region: str):
    
    # Read in the raw time series data for all NERC regions:
    temp_df = pd.read_csv((temp_data_dir + 'NERC_Region_Daily_Temperature_1980_to_2024.csv'))
    
    # Subset to just the data for NERC region you want to use:
    subset_df = temp_df[(temp_df['Region'] == temp_region)].copy()

    # Set 'Date' to a datetime variable and sort by date:
    subset_df['Time_UTC'] = pd.to_datetime(subset_df['Date'])
    subset_df = subset_df.sort_values(['Time_UTC'])

    # Add the day of year to be used as an averaging parameter:
    subset_df['DoY'] = subset_df['Time_UTC'].dt.dayofyear

    # Calculate the mean T_Min and T_Max by day of year:
    subset_df['T_Min_Mean'] = subset_df.groupby('DoY')['T_Min'].transform('mean').round(2)
    subset_df['T_Max_Mean'] = subset_df.groupby('DoY')['T_Max'].transform('mean').round(2)
    
    # Only keep the columns we need:
    output_df = subset_df[['Time_UTC','T_Min','T_Min_Mean','T_Max','T_Max_Mean']].copy()
    
    return output_df


In [ ]:
# Test the function:
temp_df = process_temperature_time_series(temp_data_dir = temp_data_dir, 
                                          temp_region = 'CA')

temp_df


## Write a Function to Process the Load Time Series Data


In [ ]:
def process_load_time_series(load_data_dir: str, load_region: str):

    # Read in the load data and subset to a given year:
    load_df = pd.read_csv((load_data_dir + 'WECC_Hourly_Loads_1980_to_2025.csv'))

    # Set 'Time_UTC' to a datetime variable:
    load_df['Time_UTC'] = pd.to_datetime(load_df['Time_UTC'])
    
    # Only keep the columns we need:
    load_df = load_df[['Time_UTC', 'WECC_Load_MWh', (load_region + '_Load_MWh')]].copy()

    # Rename the columns because I'm OCD:
    load_df.rename(columns={(load_region + '_Load_MWh'): 'Region_Load_MWh'}, inplace=True)

    # Replace empty cells with NaN and drop rows containing NaN load values:
    load_df['Region_Load_MWh'].replace('', np.nan, inplace=True)
    load_df.dropna(subset=['Region_Load_MWh'], inplace=True)
    
    # Add the hour of year to be used as an averaging parameter:
    load_df['HoY'] = (((load_df['Time_UTC'].dt.dayofyear -1) * 24) + load_df['Time_UTC'].dt.hour)
        
    # Calculate the mean load by hour of year:
    load_df['WECC_Load_Mean_MWh'] = load_df.groupby('HoY')['WECC_Load_MWh'].transform('mean').round(2)
    load_df['Region_Load_Mean_MWh'] = load_df.groupby('HoY')['Region_Load_MWh'].transform('mean').round(2)

    # Only keep the columns we need:
    output_df = load_df[['Time_UTC','WECC_Load_MWh','WECC_Load_Mean_MWh','Region_Load_MWh','Region_Load_Mean_MWh']].copy()
    
    return output_df
    

In [ ]:
# Test the function:
load_df = process_load_time_series(load_data_dir = load_data_dir, 
                                   load_region = 'CA')

load_df


## Write a Function to Process the Load Shed Time Series Data


In [ ]:
def process_load_shed_time_series(gridview_data_dir: str, load_shed_region: str):

    # Read in the load data and subset to a given year:
    load_shed_df = pd.read_csv((gridview_data_dir + 'all_' + load_shed_region + '_load_shed.csv'))

    # Rename the columns:
    load_shed_df.rename(columns={load_shed_df.columns[0]: 'Time_LT'}, inplace=True)
    load_shed_df.rename(columns={load_shed_df.columns[1]: 'Load_Shed_MWh'}, inplace=True)

    # Set 'Time_LT' to a datetime variable:
    load_shed_df['Time_LT'] = pd.to_datetime(load_shed_df['Time_LT'])

    # Convert the time to UTC:
    load_shed_df['Time_UTC'] = load_shed_df['Time_LT'] + pd.Timedelta('7 hours')
    
    # Round off the load shed values:
    load_shed_df['Load_Shed_MWh'] = load_shed_df['Load_Shed_MWh'].round(2)

    # Rearrange the columns:
    load_shed_df = load_shed_df[['Time_UTC','Load_Shed_MWh']].copy()
    
    return load_shed_df
    

In [ ]:
# Test the function:
load_shed_df = process_load_shed_time_series(gridview_data_dir = gridview_data_dir, 
                                             load_shed_region = 'CA')

load_shed_df


## Write a Function to Process the Locational Marginal Prices (LMP) Time Series Data


In [ ]:
def process_lmp_time_series(gridview_data_dir: str, lmp_region: str):

    # Read in the load data and subset to a given year:
    lmp_df = pd.read_csv((gridview_data_dir + 'all_' + lmp_region + '_lmp.csv'))

    # Rename the columns:
    lmp_df.rename(columns={lmp_df.columns[0]: 'Time_LT'}, inplace=True)
    lmp_df.rename(columns={lmp_df.columns[1]: 'LMP_Dollars_per_MW'}, inplace=True)

    # Set 'Time_LT' to a datetime variable:
    lmp_df['Time_LT'] = pd.to_datetime(lmp_df['Time_LT'])

    # Convert the time to UTC:
    lmp_df['Time_UTC'] = lmp_df['Time_LT'] + pd.Timedelta('7 hours')
    
    # Round off the LMP values:
    lmp_df['LMP_Dollars_per_MW'] = lmp_df['LMP_Dollars_per_MW'].round(2)

    # Rearrange the columns:
    lmp_df = lmp_df[['Time_UTC','LMP_Dollars_per_MW']].copy()
    
    return lmp_df
    

In [ ]:
# Test the function:
lmp_df = process_lmp_time_series(gridview_data_dir = gridview_data_dir, 
                                 lmp_region = 'CA')

lmp_df


## Write a Function to Process the Generation Time Series Data


In [ ]:
def process_generation_time_series(gridview_data_dir: str, generation_region: str):

    # Read in the load data and subset to a given year:
    gen_df = pd.read_csv((gridview_data_dir + 'all_' + generation_region + '_generation.csv'))

    # Rename the date column:
    gen_df.rename(columns={'Unnamed: 0': 'Time_LT'}, inplace=True)
    
    # Convert the time to a datetime variable:
    gen_df['Time_LT'] = pd.to_datetime(gen_df['Time_LT'])

    # Convert the time to UTC:
    gen_df['Time_UTC'] = gen_df['Time_LT'] + pd.Timedelta('7 hours')
    
    # Round off the generation values to a single decimal:
    gen_df['Coal'] = gen_df['Coal'].round(1)
    gen_df['Gas'] = gen_df['Gas'].round(1)
    gen_df['Hydro'] = gen_df['Hydro'].round(1)
    gen_df['Other'] = gen_df['Other'].round(1)
    gen_df['Solar'] = gen_df['Solar'].round(1)
    gen_df['Wind'] = gen_df['Wind'].round(1)
    gen_df['Imports'] = gen_df['Imports'].round(1)

    # Rearrange the columns:
    gen_df = gen_df[['Time_UTC','Coal','Gas','Hydro','Other','Solar','Wind','Imports']].copy()
    
    return gen_df
    

In [ ]:
# Test the function:
generation_df = process_generation_time_series(gridview_data_dir = gridview_data_dir, 
                                               generation_region = 'CA')

generation_df


## Create the Integrated Time Series by Merging the Data Streams Together


In [ ]:
# Define a function to process the integrated time series for a given NERC region:
def process_integrated_time_series(region: str, load_data_dir: str, temp_data_dir: str, gridview_data_dir: str, data_output_dir: str):

    # Homogenize the region names:
    if region == 'GB':
       load_shed_region = 'BS'
       generation_region = 'BS'
       lmp_region = 'BS'
    elif region == 'PNW':
       load_shed_region = 'NW'
       generation_region = 'NW'
       lmp_region = 'NW'
    else:
       load_shed_region = region
       generation_region = region 
       lmp_region = region 
    
    # Process the load data:
    load_df = process_load_time_series(load_data_dir = load_data_dir, load_region = region)
    load_df['Date'] = load_df['Time_UTC'].dt.date
    load_df['Date'] = pd.to_datetime(load_df['Date'])
    
    # Process the temperature data and rename the date variable:
    temp_df = process_temperature_time_series(temp_data_dir = temp_data_dir, temp_region = region)
    temp_df.rename(columns={'Time_UTC': 'Date'}, inplace=True)

    # Merge the two dataframes together based on common times:
    output_df = load_df.merge(temp_df, on=['Date'], how='left')

    # Process the load shed data:
    load_shed_df = process_load_shed_time_series(gridview_data_dir = gridview_data_dir, load_shed_region = load_shed_region)

    # Merge the load shed data into the output dataframe based on common times:
    output_df = output_df.merge(load_shed_df, on=['Time_UTC'], how='left')

    # Process the LMP data:
    lmp_df = process_lmp_time_series(gridview_data_dir = gridview_data_dir, lmp_region = lmp_region)

    # Merge the LMP data into the output dataframe based on common times:
    output_df = output_df.merge(lmp_df, on=['Time_UTC'], how='left')
    
    # Process the generation data:
    generation_df = process_generation_time_series(gridview_data_dir = gridview_data_dir, generation_region = generation_region)

    # Merge the generation data into the output dataframe based on common times:
    output_df = output_df.merge(generation_df, on=['Time_UTC'], how='left')

    # Strip the units from the column names for simplicity:
    output_df.rename(columns={'WECC_Load_MWh': 'WECC_Load', 
                              'WECC_Load_Mean_MWh': 'WECC_Load_Mean',
                              'Region_Load_MWh': 'Region_Load',
                              'Region_Load_Mean_MWh': 'Region_Load_Mean',
                              'LMP_Dollars_per_MW': 'LMP',
                              'Load_Shed_MWh': 'Load_Shed'}, inplace=True) 
    
    # Rearrange the columns:
    output_df = output_df[['Time_UTC','T_Min','T_Min_Mean','T_Max','T_Max_Mean','WECC_Load','WECC_Load_Mean','Region_Load','Region_Load_Mean',
                           'Coal','Gas','Hydro','Other','Solar','Wind','Imports','LMP','Load_Shed']].copy()

    # Set the output filename:
    output_filename = (region + '_Integrated_Time_Series_1980_to_2024.csv')
        
    # Write out the dataframe to a .csv file:
    output_df.to_csv((os.path.join(data_output_dir, output_filename)), sep=',', index=False)
    
    return output_df


In [ ]:
# Test the function:
integrated_df = process_integrated_time_series(region = 'PNW',
                                               load_data_dir = load_data_dir,
                                               temp_data_dir = temp_data_dir,
                                               gridview_data_dir = gridview_data_dir,
                                               data_output_dir = data_output_dir)

integrated_df


In [ ]:
# Loop over the NERC TPL-008-1 regions in the WECC and process the time series for each one:
for region in ['CA', 'GB', 'PNW', 'RM', 'SW']:
    process_integrated_time_series(region = region,
                                   load_data_dir = load_data_dir,
                                   temp_data_dir = temp_data_dir,
                                   gridview_data_dir = gridview_data_dir,
                                   data_output_dir = data_output_dir)
